<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c39-header.png" alt="Nextia Learning · Entity Resolution and Record Matching: Is This the Same Thing?" width="100%">

# Link bibliographic records

**[Link bibliographic records](https://learning.nextia-ai.com/courses/record-matching/m09/link-bibliographic-records/)** · Entity Resolution and Record Matching: Is This the Same Thing? · Module 9, case study 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** link real bibliographic records from two libraries from start to end: normalise titles, authors and venues, block, compare, classify, compare methods under one measure, study the errors, and choose a design with its thresholds and its review band.

| | |
|---|---|
| **Time** | About 60 minutes in the notebook (the lesson page has the full story) |
| **Needs** | An internet connection for the setup cells (about 2 MB) |
| **You should know** | The course's steps, from [A supervised pair classifier](https://learning.nextia-ai.com/courses/record-matching/m06/a-supervised-pair-classifier/) and [Human review](https://learning.nextia-ai.com/courses/record-matching/m08/human-review/) |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, rapidfuzz 3.14.6, scikit-learn 1.9.1, LightGBM 4.7.0, Splink 5.0.0 and DuckDB 2.0.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/record-matching/m09/link-bibliographic-records/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/M09-L02-link-bibliographic-records/link-bibliographic-records-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, rapidfuzz 3.14.6, scikit-learn 1.9.1, lightgbm 4.7.0, splink 5.0.0, duckdb 2.0.0, pyarrow 26.0.1, and matplotlib. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 rapidfuzz==3.14.6 scikit-learn==1.9.1 lightgbm==4.7.0 splink==5.0.0 duckdb==2.0.0 pyarrow==26.0.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'rapidfuzz', 'scikit-learn', 'lightgbm', 'splink', 'duckdb', 'pyarrow']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It downloads one small zip file (270 KB) from the benchmark's own page at the University of Leipzig, and the shipped embeddings (about 2 MB) from the course's labs repository, into a folder `bibliographic-records`. It checks every file's checksum. It needs an internet connection and no account.

You should see `Ready:` and the folders. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `bibliographic-records`, then run the cell again. If it happens again, the file on the source page has changed: the notebook stops, so that it never runs on other data than the lesson.

**Source and licence.** "Benchmark datasets for entity resolution" by the database group of Prof. Erhard Rahm, University of Leipzig (https://dbs.uni-leipzig.de/research/projects/benchmark-datasets-for-entity-resolution), under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Paper: H. Köpcke, A. Thor and E. Rahm, "Evaluation of entity resolution approaches on real-world match problems", PVLDB 3(1), 2010. The shipped embeddings are made from these titles by Nextia Learning; nothing else was changed.

In [ ]:
# Setup: download the benchmark from its source and the shipped embeddings from labs, and check every checksum.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

SOURCE = ("https://dbs.uni-leipzig.de/files/datasets/DBLP-ACM.zip", "e37e7ed8e06722499e6d9e94583fdec279207b13ce6270a37595b6a5d594bc40")   # the zip file on the Leipzig benchmark page, and its SHA-256
LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C39/M09-L02-link-bibliographic-records/"
FILES = {
    "data/embeddings/dblp_int8.npy": "cd6742a95d72e6f4fdd5e82a9bd3f283d70ad1c7f4ed5c9d533cbdb51c0d83b1",
    "data/embeddings/dblp_scale.npy": "1cd6555a60910c4e92a5948660185f9c85b8de09f07d0925c3f57f400ee681b4",
    "data/embeddings/acm_int8.npy": "ebf273d2eb304e85b3480bed2b42a8022e6557ca49936791a0f61f15ffbb8e60",
    "data/embeddings/acm_scale.npy": "566cc1c2825000f67d88de73649673277e9f9124bc2efc03226ad450df185c7e",
    "data/embeddings/dblp_ids.csv": "e405b14aba44a82e95f3f75146df4794d6d0117bcc712221a7bf21d7377d26ac",
    "data/embeddings/acm_ids.csv": "6f5828cd4eebc3c860d853632ea01f20f1ccaa5bcc9ca40d8d7d7ccffd1e180f",
    "data/embeddings/model.json": "c8bb089aa55e07659e46e0c8ae0be2d8ee0e544eadcc82947c8658e4ca525e5b"
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("bibliographic-records").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, expected):
    path = WORK / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        shutil.copy(local, path) if local.exists() else download(url, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder bibliographic-records and run again.")
    return path

with zipfile.ZipFile(fetch("data/DBLP-ACM.zip", *SOURCE)) as z:
    z.extractall(WORK / "data" / "dblp-acm")
for name, expected in FILES.items():
    fetch(name, LABS + name, expected)
os.chdir(WORK)
print("Ready:", sorted(p.name for p in Path("data").iterdir()))

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd` and matplotlib. It defines:

- `load()` reads the two libraries and their true links; `split_records()` puts every record in train, validation or test.
- `with_labels()` marks each pair as a true link or not; `pair_measures()` gives pair precision, recall and F1 on one split; `blocking_measures()` gives pair completeness and reduction ratio.
- The check helpers `expect()` and `expect_hash()`: they say if your answer is right, without showing the answer.

Run the cell. It prints nothing.

In [ ]:
# Helpers: load the two sources, split, measure, and checks that do not show the answer.
import hashlib, html, json, re, time, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 39

def load():
    """DBLP (source A), ACM (source B) and the true matches (id_a, id_b). The files are Latin-1 text."""
    read = lambda f: pd.read_csv(f"data/dblp-acm/{f}", encoding="latin-1", dtype=str, keep_default_na=False)
    a, b, m = read("DBLP2.csv"), read("ACM.csv"), read("DBLP-ACM_perfectMapping.csv")
    m.columns = ["id_a", "id_b"]
    return a, b, m

def split_records(a, b, m, shares=(0.5, 0.25, 0.25), seed=SEED):
    """train / validation / test for every record, by group: the records joined by true matches stay together."""
    parent = {}
    def find(x):
        while parent.setdefault(x, x) != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for x in list("a" + a.id) + list("b" + b.id):
        find(x)
    for x, y in zip("a" + m.id_a, "b" + m.id_b):
        parent[find(x)] = find(y)
    groups = sorted({find(x) for x in parent})
    order = np.random.default_rng(seed).permutation(len(groups))
    cut1, cut2 = int(shares[0] * len(groups)), int((shares[0] + shares[1]) * len(groups))
    names = np.empty(len(groups), dtype=object)
    names[order[:cut1]], names[order[cut1:cut2]], names[order[cut2:]] = "train", "validation", "test"
    of_group = dict(zip(groups, names))
    return {x: of_group[find(x)] for x in parent}

def with_labels(pairs, m, split):
    """Adds `same` (a true match) and `split` (the split of the DBLP record) to a table of pairs."""
    true = set(zip(m.id_a, m.id_b))
    return pairs.assign(same=[(x, y) in true for x, y in zip(pairs.id_a, pairs.id_b)],
                        split=("a" + pairs.id_a).map(split).to_numpy())

def pair_measures(called, m, split, which="test"):
    """Pair precision, recall and F1 on one split: `called` is the pairs (id_a, id_b) a matcher calls the same."""
    true = {(x, y) for x, y in zip(m.id_a, m.id_b) if split["a" + x] == which}
    c = {(x, y) for x, y in zip(called.id_a, called.id_b) if split["a" + x] == which}
    right = len(c & true)
    p = right / len(c) if c else 0.0
    r = right / len(true) if true else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"precision": round(p, 3), "recall": round(r, 3), "f1": round(f1, 3), "called": len(c), "right": right,
            "true": len(true)}

def blocking_measures(pairs, m, n_a, n_b):
    """Pair completeness (true matches kept) and reduction ratio (share of all pairs removed) of a blocking."""
    true = set(zip(m.id_a, m.id_b))
    kept = sum((x, y) in true for x, y in zip(pairs.id_a, pairs.id_b))
    return {"candidates": len(pairs), "true kept": kept, "pair completeness": round(kept / len(true), 4),
            "reduction ratio": round(1 - len(pairs) / (n_a * n_b), 5)}

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

A university's research office must list every paper that its researchers published, for a yearly report. It takes the records from two bibliographic libraries: **DBLP**, a free computer-science index, and the **ACM Digital Library**. Many papers are in both. The office must know which DBLP record and which ACM record describe **the same paper**.

Each error has a price. A **false link** joins two different papers: one paper disappears from the report. A **missed link** counts one paper twice. The office links automatically only when almost every link is right; a person checks the unsure ones.

You will build that linker with the course's steps (normalise → block → compare → classify → measure), compare several methods under one measure, study the errors, and choose a design with its thresholds and its review band.

## 2. The data

The benchmark comes from the database group of the University of Leipzig, under the CC BY 4.0 licence. The setup cell downloaded it from the source page. One row is one paper as one library lists it: title, authors, venue (the conference or journal) and year. The third file lists the **true links**.

In [ ]:
dblp, acm, matches = load()
print("DBLP:", dblp.shape, dblp.columns.tolist())
print("ACM: ", acm.shape, acm.columns.tolist())
print("true matches:", len(matches))

> **Check.** DBLP has 2,616 records and ACM 2,294, with 2,224 true links. Both files have the same five columns.

Before any method, check the gaps and the repeats:

In [ ]:
print("repeated IDs:", int(dblp.id.duplicated().sum()), int(acm.id.duplicated().sum()))
print("no authors:", f"DBLP {(dblp.authors.str.strip() == '').sum()},", f"ACM {(acm.authors.str.strip() == '').sum()}")
print("records with a match:", f"DBLP {dblp.id.isin(matches.id_a).sum()} of {len(dblp)},",
      f"ACM {acm.id.isin(matches.id_b).sum()} of {len(acm)}")
print("records with two or more matches:", f"DBLP {(matches.id_a.value_counts() > 1).sum()},",
      f"ACM {(matches.id_b.value_counts() > 1).sum()}")
pairs_true = (matches.merge(dblp.add_suffix("_a"), on="id_a").merge(acm.add_suffix("_b"), on="id_b"))
print("true matches with the same year:", f"{(pairs_true.year_a == pairs_true.year_b).mean():.0%}")
print("true matches with the same title, as written:", f"{(pairs_true.title_a == pairs_true.title_b).mean():.0%}")
print("true matches with the same author list, as written:", f"{(pairs_true.authors_a == pairs_true.authors_b).mean():.0%}")

> **Check.** The year is the same in every true link. But the title is the same, as written, in only 41% of them, and the author list in only 28%. Also, not every record has a partner: 392 DBLP records and 70 ACM records have none.

Four true links, side by side:

In [ ]:
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 200)
shown = pairs_true.sample(4, random_state=7)
for _, r in shown.iterrows():
    print(f"DBLP: {r.title_a} | {r.authors_a} | {r.venue_a} | {r.year_a}")
    print(f"ACM:  {r.title_b} | {r.authors_b} | {r.venue_b} | {r.year_b}\n")

The capitals differ, ACM writes the authors in another order, and ACM writes letters such as ö as HTML codes (`&#246;`). The venue has a short name in DBLP and a long one in ACM:

In [ ]:
print(pd.crosstab(pairs_true.venue_a, pairs_true.venue_b.str[:30]).to_string())

> **Check.** Each DBLP venue goes with exactly one ACM venue. So you can write a small table that gives both names one key.

## 3. Split and baseline

The model will learn from labelled pairs, so you need a fair test. The split puts each record in **train**, **validation** or **test**, by group: records joined by a true link stay in one split. A pair belongs to the split of its DBLP record. You choose every threshold on validation, and you look at test only to report.

In [ ]:
split = split_records(dblp, acm, matches)
true_in = pd.Series([split["a" + x] for x in matches.id_a]).value_counts()
print("true matches per split:", true_in.to_dict())
print("all pairs:", f"{len(dblp) * len(acm):,}")

The simplest linker: the same title, as written, in both libraries.

In [ ]:
same_title = dblp.merge(acm, on="title", suffixes=("_a", "_b"))[["id_a", "id_b"]]
print("exact title, as written:", pair_measures(same_title, matches, split))

> **Check.** Precision 0.909, but recall only 0.419: capitals and small signs hide most links.

## 4. Normalise titles, authors and venues

`normalise()` decodes the HTML codes, writes lower case, drops accents and signs. `surnames()` keeps the last word of each author's name. `VENUE_KEYS` gives both names of a venue one key.

In [ ]:
LETTERS = str.maketrans({"þ": "th", "ð": "d", "ß": "ss", "æ": "ae", "ø": "o", "ł": "l"})   # letters with no accent to drop

def normalise(text):
    """HTML codes decoded, lower case, no accents, no signs, one space between words."""
    s = html.unescape(str(text)).lower()                                   # Bj&#246;rn -> björn
    s = s.translate(LETTERS).replace("'", "")                              # þór -> thor, editor's -> editors
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    s = re.sub(r"(?<=\w)[-/](?=\w)", "", s)                                 # object-oriented -> objectoriented
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return " ".join(s.split())

VENUE_KEYS = {  # one short key per venue, for both sources (written by hand from the table above)
    "SIGMOD Conference": "sigmod", "International Conference on Management of Data": "sigmod",
    "SIGMOD Record": "sigmod record", "ACM SIGMOD Record": "sigmod record",
    "VLDB": "vldb", "Very Large Data Bases": "vldb",
    "VLDB J.": "vldb journal", "The VLDB Journal &mdash; The International Journal on Very Large Data Bases": "vldb journal",
    "ACM Trans. Database Syst.": "tods", "ACM Transactions on Database Systems (TODS)": "tods",
}

def surnames(authors):
    """The set of surnames (the last word of each name) in an author list."""
    return {n.split()[-1] for n in (normalise(x) for x in str(authors).split(",")) if n}

for t in ["Bj&#246;rn Th&#243;r J&#243;nsson, Michael J. Franklin", "Björn Þór Jónsson, Michael J. Franklin",
          "The &#961; operator: discovering and ranking associations", "PESTO : An Integrated Query/Browser for Object Databases"]:
    print(f"{t}\n   -> {normalise(t)}")
print(surnames("Donald Kossmann, Michael J. Franklin, Björn Þór Jónsson"))

In [ ]:
def prepare(df):
    """One row per record: the normalised title, the author surnames, the number of authors, the year and the venue key."""
    return pd.DataFrame({"id": df.id, "title": df.title.map(normalise), "authors": df.authors.map(normalise),
                         "surnames": df.authors.map(surnames),
                         "n_authors": df.authors.map(lambda s: len([x for x in s.split(",") if x.strip()])),
                         "year": df.year.astype(int), "venue": df.venue.str.strip().map(VENUE_KEYS)})

A, B = prepare(dblp), prepare(acm)
print(A.head(3).to_string())
print("venues without a key:", int(A.venue.isna().sum()), int(B.venue.isna().sum()))

> **Predict.** A rule says "same paper when the normalised titles are the same". A second rule also asks for the same year and venue. Write down which one has the higher precision before you run the next cell.

In [ ]:
def same_values(A, B, cols):
    """Every pair of records with the same values in the columns."""
    return A[["id", *cols]].merge(B[["id", *cols]], on=cols, suffixes=("_a", "_b"))[["id_a", "id_b"]]

rule_title = same_values(A, B, ["title"])
rule_tyv = same_values(A, B, ["title", "year", "venue"])
print("same normalised title:            ", pair_measures(rule_title, matches, split))
print("same normalised title, year, venue:", pair_measures(rule_tyv, matches, split))

> **Check.** On clean data, a rule already does well: title, year and venue reach F1 0.933 (precision 0.974). The year and the venue remove most false links of the title alone.

### Failure case: a title that many papers share

In [ ]:
repeated = A.title.value_counts()
print("the most common titles in DBLP:", repeated.head(4).to_dict())
wrong = with_labels(rule_title, matches, split)
wrong = wrong[~wrong.same].merge(A[["id", "title"]], left_on="id_a", right_on="id")
print("false matches of 'same title':", len(wrong), "| from the 4 most common titles:", int(wrong.title.isin(repeated.head(4).index).sum()))

> **Warning.** "Editor's notes" or "Reminiscences on influential papers" is the title of a regular column, not of one paper. Each such title joins every record of one library to every record of the other. Before you trust a key, look at its most common values.

## 5. Blocking

All pairs of the two libraries are 6,001,104. You keep only candidate pairs, with three blockings to compare:

- **same year and venue**: safe here, because every true link has the same year and venue;
- **TF-IDF top 5**: for each record, the 5 records of the other library with the most similar title (TF-IDF compares the short pieces of letters that two titles share, and gives rare pieces more weight);
- **embeddings top 5**: the same with the cosine of the titles' embeddings (the shipped embeddings of the course's Granite model).

In [ ]:
def load_embeddings(source, ids):
    """The shipped embeddings of one source's titles (8-bit numbers times one scale per row), in the order of ids."""
    e = np.load(f"data/embeddings/{source}_int8.npy").astype("float32") * np.load(f"data/embeddings/{source}_scale.npy")
    row = {r: i for i, r in enumerate(pd.read_csv(f"data/embeddings/{source}_ids.csv", dtype=str).id)}
    e = e[[row[x] for x in ids]]
    return e / np.linalg.norm(e, axis=1, keepdims=True)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True).fit(pd.concat([A.title, B.title]))
S_tfidf = (tfidf.transform(A.title) @ tfidf.transform(B.title).T).toarray()
S_embed = load_embeddings("dblp", A.id) @ load_embeddings("acm", B.id).T
print(S_tfidf.shape, S_embed.shape)

In [ ]:
def top_k(S, A, B, k):
    """For each record of A its k most similar records of B, and for each record of B its k most similar of A."""
    ia = np.argsort(-S, axis=1)[:, :k]
    ib = np.argsort(-S, axis=0)[:k, :]
    p1 = pd.DataFrame({"id_a": np.repeat(A.id.to_numpy(), k), "id_b": B.id.to_numpy()[ia.ravel()]})
    p2 = pd.DataFrame({"id_a": A.id.to_numpy()[ib.T.ravel()], "id_b": np.repeat(B.id.to_numpy(), k)})
    return pd.concat([p1, p2]).drop_duplicates().reset_index(drop=True)

K = 5
blockings = {"same year and venue": same_values(A, B, ["year", "venue"]),
             f"TF-IDF top {K}": top_k(S_tfidf, A, B, K), f"embeddings top {K}": top_k(S_embed, A, B, K)}
blockings["TF-IDF and embeddings"] = pd.concat([blockings[f"TF-IDF top {K}"], blockings[f"embeddings top {K}"]]).drop_duplicates().reset_index(drop=True)
blocking_table = pd.DataFrame({k: blocking_measures(v, matches, len(A), len(B)) for k, v in blockings.items()}).T
print(blocking_table.to_string())

> **Check.** "Same year and venue" keeps every true link, but with about 158,000 candidates. TF-IDF and embeddings together keep 2,221.0 of the 2,224 true links with only 28,561 candidates. You use them.

> **Your turn.** Use `k_small = 1` instead of 5: only the single most similar record, both ways. Compute the union of the two top-k blockings, and its measures.

In [ ]:
k_small = 1
small = ...            # the union of top_k(S_tfidf, ...) and top_k(S_embed, ...) with k_small, without repeats
small_measures = blocking_measures(small, matches, len(A), len(B)) if small is not ... else None
print(small_measures)

In [ ]:
expect_hash('small_measures', small_measures, '2409977d8e5ed633')

> **Check.** With one neighbour, the blocking still keeps 99% of the true links. Clean titles make blocking easy.

## 6. Compare

Each candidate pair gets a **comparison vector**: three title similarities, three author similarities (the shared surnames, the author strings in any order, the difference in the number of authors), the year difference, "same venue", and the embedding cosine. A missing author list stays missing.

In [ ]:
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

def compare(pairs, A, B, S_tfidf, S_embed):
    """The comparison vector of each candidate pair: one similarity per piece of evidence (missing = NaN)."""
    a = A.set_index("id").loc[pairs.id_a].reset_index(drop=True)
    b = B.set_index("id").loc[pairs.id_b].reset_index(drop=True)
    ra = pairs.id_a.map(dict(zip(A.id, range(len(A))))).to_numpy()
    rb = pairs.id_b.map(dict(zip(B.id, range(len(B))))).to_numpy()
    v = pd.DataFrame(index=pairs.index)
    v["title_tsr"] = [fuzz.token_set_ratio(x, y) / 100 for x, y in zip(a.title, b.title)]
    v["title_jw"] = [JaroWinkler.similarity(x, y) for x, y in zip(a.title, b.title)]
    v["tfidf"] = S_tfidf[ra, rb]
    v["authors_jac"] = [len(x & y) / len(x | y) if x and y else np.nan for x, y in zip(a.surnames, b.surnames)]
    v["authors_tsr"] = [fuzz.token_sort_ratio(x, y) / 100 if x and y else np.nan for x, y in zip(a.authors, b.authors)]
    v["n_authors_diff"] = [abs(x - y) if x and y else np.nan for x, y in zip(a.n_authors, b.n_authors)]
    v["year_diff"] = (a.year - b.year).abs().to_numpy()
    v["venue_eq"] = (a.venue == b.venue).astype(float).to_numpy()
    v["embed_cos"] = S_embed[ra, rb]
    return v

t0 = time.perf_counter()
cand = with_labels(blockings["TF-IDF and embeddings"], matches, split)
vectors = compare(cand, A, B, S_tfidf, S_embed)
compare_seconds = time.perf_counter() - t0
print(f"{len(cand):,} candidate pairs, {cand.same.sum():,} true; {compare_seconds:.1f} s")
print(vectors[cand.same].head(2).round(3).to_string())

## 7. Classify, and compare the methods under one measure

Every method is measured the same way: on the test split, at the threshold with the best F1 on validation. A second measure fits the office's rule: the **precision at recall 0.95**, at the threshold that found 95% of the validation links. The first rows have no training: rules, and one similarity alone.

In [ ]:
def measures_at(score, threshold, which="test"):
    return pair_measures(cand[(score >= threshold) & (cand.split == which).to_numpy()], matches, split, which)

def choose_threshold(score, goal="f1", recall=0.95):
    """A threshold from the validation split: the best F1, or the highest threshold that reaches a recall (None if
    no threshold reaches it)."""
    best, best_t = -1, None
    for t in np.unique(np.round(score[(cand.split == "validation").to_numpy()], 4))[::-1]:
        m_ = measures_at(score, t, "validation")
        if goal == "f1" and m_["f1"] > best:
            best, best_t = m_["f1"], t
        if goal == "recall" and m_["recall"] >= recall:
            return t
    return best_t if goal == "f1" else None

results = {}
def add_result(name, score, threshold=None):
    """One row of the comparison table, on the test split: the measures at the best-F1 threshold (or a fixed one
    for a rule), and the precision at the threshold that reached recall 0.95 on validation."""
    t = choose_threshold(score) if threshold is None else threshold
    t95 = choose_threshold(score, "recall", 0.95) if threshold is None else None
    at95 = measures_at(score, t95) if t95 is not None else {"precision": None, "recall": None}
    results[name] = {**measures_at(score, t), "threshold": round(float(t), 3),
                     "precision at recall 0.95": at95["precision"], "recall there": at95["recall"]}
    return results[name]

key = lambda d: set(zip(d.id_a, d.id_b))
def in_rule(rule):
    """1 for a candidate pair that the rule calls the same, else 0."""
    called = key(rule)
    return np.array([p in called for p in zip(cand.id_a, cand.id_b)], dtype=float)
add_result("exact title, as written", in_rule(same_title), threshold=1.0)
add_result("rule: same title", in_rule(rule_title), threshold=1.0)
add_result("rule: same title, year and venue", in_rule(rule_tyv), threshold=1.0)
add_result("TF-IDF alone", vectors.tfidf.to_numpy())
add_result("embeddings alone", vectors.embed_cos.to_numpy())
print(pd.DataFrame(results).T.to_string())

Splink learns its weights **without labels**, as in Module 5. It compares the title (Jaro-Winkler levels), the shared surnames, the year and the venue.

In [ ]:
import logging, duckdb, splink.comparison_library as cl
from splink import Linker, SettingsCreator, block_on
from splink.backends.duckdb import DuckDBAPI
Path("duckdb_tmp").mkdir(exist_ok=True)
con = duckdb.connect()   # DuckDB may write temporary files: keep them in one folder, at most 4 GB
con.execute("SET threads = 1; SET memory_limit = '2GB'; SET temp_directory = 'duckdb_tmp'; SET max_temp_directory_size = '4GB'")

def splink_table(P):
    return pd.DataFrame({"unique_id": P.id, "title": P.title, "surnames": P.surnames.map(sorted),
                         "year": P.year, "venue": P.venue})

settings = SettingsCreator(
    link_type="link_only", unique_id_column_name="unique_id",
    comparisons=[cl.JaroWinklerAtThresholds("title", [0.95, 0.85]), cl.ArrayIntersectAtSizes("surnames", [3, 2, 1]),
                 cl.ExactMatch("year"), cl.ExactMatch("venue")],
    blocking_rules_to_generate_predictions=[block_on("year")])
t0 = time.perf_counter()
api = DuckDBAPI(connection=con)
linker = Linker([api.register(splink_table(A)), api.register(splink_table(B))], settings)
for name in ("splink", "splink.internals"):   # Splink prints every training step: show only warnings
    logging.getLogger(name).setLevel(logging.WARNING)
linker.training.estimate_probability_two_random_records_match([block_on("title")], recall=0.8)
linker.training.estimate_u_using_random_sampling(max_pairs=1e7, seed=1)   # all 6 million pairs
linker.training.estimate_parameters_using_expectation_maximisation(block_on("year", "venue"))
linker.training.estimate_parameters_using_expectation_maximisation(block_on("substr(title, 1, 12)"))
splink_seconds = time.perf_counter() - t0
scored = linker.inference.predict().as_pandas_dataframe()
weight = dict(zip(zip(scored.unique_id_l, scored.unique_id_r), scored.match_weight))
splink_score = np.array([weight.get((x, y), -99.0) for x, y in zip(cand.id_a, cand.id_b)])
print(f"trained without labels in {splink_seconds:.1f} s;", add_result("Splink, no labels", splink_score))

> **Predict.** On products, Splink learned a much too low m probability for the model codes. Here, what m do you expect for "exact match on title"?

In [ ]:
levels = [(c["output_column_name"], l["label_for_charts"], l["m_probability"], l["u_probability"])
          for c in linker.misc.save_model_to_json()["comparisons"] for l in c["comparison_levels"] if not l.get("is_null_level")]
splink_levels = pd.DataFrame(levels, columns=["field", "level", "m", "u"]).round(4)
print(splink_levels.to_string(index=False))

> **Check.** Splink learned sensible values: most true links agree on the title, the year and the venue. With clean, separate fields, expectation maximisation finds the right two groups. Splink without labels reaches F1 0.957, better than the best rule.

Now the classifiers that learn from the labelled training pairs:

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train = (cand.split == "train").to_numpy()
FIELDS = [c for c in vectors.columns if c != "embed_cos"]
def flat(x):   # logistic regression needs numbers: missing -> 0, plus a "missing" column
    return pd.concat([x.fillna(0), x.isna().astype(float).add_suffix("_missing")], axis=1)
def new_lgbm():
    return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=20, random_state=1,
                          n_jobs=1, verbose=-1)

t0 = time.perf_counter()
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(flat(vectors[FIELDS])[train], cand.same[train])
lgbm = new_lgbm().fit(vectors[FIELDS][train], cand.same[train])
lgbm_score = lgbm.predict_proba(vectors[FIELDS])[:, 1]
learn_seconds = time.perf_counter() - t0
logreg_score = logreg.predict_proba(flat(vectors[FIELDS]))[:, 1]
add_result("logistic regression", logreg_score)
add_result("LightGBM", lgbm_score)
add_result("LightGBM + embedding cosine", new_lgbm().fit(vectors[train], cand.same[train]).predict_proba(vectors)[:, 1])
print(f"trained on {train.sum():,} labelled pairs ({cand.same[train].sum()} true) in {learn_seconds:.1f} s")
print(pd.DataFrame(results).T[["precision", "recall", "f1", "precision at recall 0.95"]].to_string())

> **Check.** LightGBM: F1 0.981 (precision 0.971, recall 0.991).

Which evidence does LightGBM use?

In [ ]:
gain = pd.Series(lgbm.booster_.feature_importance("gain"), index=FIELDS)
print((gain / gain.sum()).round(3).sort_values(ascending=False).to_string())

> **Check.** Almost all the gain is on the title. The authors get less than 2%. Keep that in mind for the next step.

## 8. Error analysis

A good F1 hides the errors. Look at each wrong decision of LightGBM on the test split, and give it a reason that you can see in the records.

In [ ]:
# Error analysis: every wrong decision of a model on the test split, with the reason we can see in the records.
title_count = pd.concat([A.title, B.title]).value_counts()   # how many records, in both sources, have each title
has_partner = set(matches.id_a)
A_i, B_i = A.set_index("id"), B.set_index("id")

def reason(row):
    v = vectors.loc[row.name]
    if title_count[A_i.title[row.id_a]] > 2:
        return "a title that many papers share"
    if not row.same and row.id_a not in has_partner:
        return "no partner in ACM"
    if v.title_tsr < 0.9:
        return "the title differs"
    if not (v.authors_jac >= 0.5):
        return "the author lists differ"
    return "other"

def error_analysis(score, threshold):
    c = cand.assign(score=score)
    c = c[c.split == "test"]
    errors = pd.concat([c[(c.score >= threshold) & ~c.same].assign(kind="false match"),
                        c[(c.score < threshold) & c.same].assign(kind="missed match")])
    errors["reason"] = errors.apply(reason, axis=1)
    return errors

errors = error_analysis(lgbm_score, results["LightGBM"]["threshold"])
test_true = {p for p in key(matches) if split["a" + p[0]] == "test"}
print("lost by blocking:", len(test_true - key(cand)))
error_table = pd.crosstab(errors.reason, errors.kind)
print(error_table.to_string())

In [ ]:
names_a, names_b = dblp.set_index("id"), acm.set_index("id")
for _, r in errors.sort_values(["kind", "reason"]).groupby(["kind", "reason"]).head(1).iterrows():
    x, y = names_a.loc[r.id_a], names_b.loc[r.id_b]
    print(f"{r.kind}, {r.reason} (score {r.score:.2f})")
    print(f"   DBLP: {x.title} | {x.authors} | {x.venue} {x.year}")
    print(f"   ACM:  {y.title} | {y.authors} | {y.venue} {y.year}")

> **Check.** Most false links come from titles that many papers share. ACM lists "Reminiscences on influential papers" under the column's editor, while DBLP lists the authors of that part. The model trusts the title, which is the same.

The fix: tell the model **how common the title is**. A new field `title_shared` counts the records, in both libraries, that have the same title. Then the model can learn "a common title is weak evidence".

In [ ]:
def title_shared(pairs, A, B, counts):
    """For each pair: how many records, in both sources, share the more common of its two titles."""
    a = pairs.id_a.map(A.set_index("id").title).map(counts).fillna(1).to_numpy()
    b = pairs.id_b.map(B.set_index("id").title).map(counts).fillna(1).to_numpy()
    return np.maximum(a, b)

vectors["title_shared"] = title_shared(cand, A, B, title_count)
FIELDS2 = FIELDS + ["title_shared"]
lgbm2 = new_lgbm().fit(vectors[FIELDS2][train], cand.same[train])
lgbm2_score = lgbm2.predict_proba(vectors[FIELDS2])[:, 1]
print(add_result("LightGBM + how common the title is", lgbm2_score))
errors2 = error_analysis(lgbm2_score, results["LightGBM + how common the title is"]["threshold"])
error_table2 = pd.crosstab(errors2.reason, errors2.kind)
print(error_table2.to_string())

> **Check.** Precision 1.0, recall 0.98: no false link left in the test split. The errors moved: the model now misses some links with a common title instead of making false ones. For the office, a missed link costs less.

> **Your turn.** Can a rule do the same? Keep only the pairs of `rule_tyv` that share at least one surname. **Predict** the F1 first.

In [ ]:
def share_a_surname(pairs):
    return np.array([bool(A_i.surnames[x] & B_i.surnames[y]) for x, y in zip(pairs.id_a, pairs.id_b)])

rule_tyv_s = ...       # the pairs of rule_tyv that share at least one surname: use share_a_surname()
rule_s_f1 = pair_measures(rule_tyv_s, matches, split)["f1"] if rule_tyv_s is not ... else None
print(rule_s_f1)

In [ ]:
expect('rule_s_f1', rule_s_f1, 0.932, tolerance=0.001)

> **Check.** Almost no change. The false links of the rule share a surname too: the column editor appears in many records, and all "Book Review Column" records have the same author. A title, a year, a venue and an author cannot tell two issues of one column apart. Only the issue number could, and neither library gives it.

## 9. One record, one partner

From the first case study: a paper is in each library at most once, so keep a pair only when it is the best pair of both its records.

In [ ]:
def one_to_one(pairs, score):
    """Keep a pair only when it has the best score of both its records (the other pairs get score 0)."""
    d = pairs.assign(s=score)
    keep = (d.s == d.groupby("id_a").s.transform("max")) & (d.s == d.groupby("id_b").s.transform("max"))
    return np.where(keep, score, 0.0)

print(add_result("LightGBM, one-to-one", one_to_one(cand, lgbm_score)))
final_score = one_to_one(cand, lgbm2_score)
print(add_result("LightGBM + how common the title is, one-to-one", final_score))

> **Check.** One-to-one helps the first LightGBM a little. After the fix, it changes nothing: the fixed model makes no false link for it to remove. It stays in the design as a cheap guard.

> **Your turn.** Train a logistic regression with the new field (`FIELDS2`), as in the LEARN cell, and add it to the table.

In [ ]:
logreg2 = ...          # a new logistic regression, as in the LEARN cell, trained on flat(vectors[FIELDS2])
logreg2_f1 = (add_result("logistic regression + how common the title is", logreg2.predict_proba(flat(vectors[FIELDS2]))[:, 1])["f1"]
              if logreg2 is not ... else None)
print(logreg2_f1)

In [ ]:
expect('logreg2_f1', logreg2_f1, 0.988, tolerance=0.001)

## 10. Decide: accept, review or no match

The office links automatically only when at least 995 in 1,000 links are right (our assumption for this case study). Pairs between a lower and an upper threshold go to a person. The table is on the validation split:

In [ ]:
def band_table(score, lower, upper, which="validation"):
    """Three bands: accept automatically (score >= upper), review (lower <= score < upper), reject the rest."""
    rows = (cand.split == which).to_numpy()
    s = cand[rows].assign(score=score[rows])
    acc, rev = s[s.score >= upper], s[(s.score >= lower) & (s.score < upper)]
    true_n = sum(split["a" + x] == which for x in matches.id_a)
    return {"accepted": len(acc), "accepted right": int(acc.same.sum()), "precision accepted": round(float(acc.same.mean()), 4),
            "to review": len(rev), "true in review": int(rev.same.sum()),
            "recall if reviews are right": round(float((acc.same.sum() + rev.same.sum()) / true_n), 4)}

grid = pd.DataFrame({(lo, up): band_table(final_score, lo, up) for lo in (0.01, 0.05, 0.2) for up in (0.5, 0.8, 0.95)}).T
grid.index.names = ["lower", "upper"]
print(grid.to_string())

In [ ]:
TARGET = 0.995               # the office's rule: at least 995 in 1,000 automatic links right (an assumption)
UPPER = min(up for (lo, up), row in grid.iterrows() if row["precision accepted"] >= TARGET)
LOWER = 0.05
decision = {w: band_table(final_score, LOWER, UPPER, w) for w in ("validation", "test")}
print("upper", UPPER, "lower", LOWER)
print(pd.DataFrame(decision).T.to_string())

> **Check.** Upper threshold 0.5, lower 0.05. On test, 539 links are accepted (539 right) and 10 pairs go to review.

What does it cost? The prices are assumptions (the course's price table): a CPU server at US$0.17 an hour, and a reviewer at 1.2 minutes a pair and US$40 an hour.

In [ ]:
costs = {"CPU server, US$ an hour": 0.17, "review minutes a pair": 1.2, "reviewer US$ an hour": 40}   # assumptions
cpu_seconds = compare_seconds + learn_seconds
per_1000_cpu = cpu_seconds / len(cand) * 1000 / 3600 * costs["CPU server, US$ an hour"]
test_records = sum(split["a" + x] == "test" for x in dblp.id)
review_per_1000 = decision["test"]["to review"] / test_records * 1000
review_hours = review_per_1000 * costs["review minutes a pair"] / 60
print(f"CPU: {cpu_seconds:.1f} s for {len(cand):,} pairs -> US${per_1000_cpu:.7f} per 1,000 candidate pairs")
print(f"Review: {review_per_1000:.0f} pairs per 1,000 DBLP records -> {review_hours:.2f} hours, US${review_hours * costs['reviewer US$ an hour']:.0f}")

## 11. The whole design on new input

`link_records()` runs every step of the chosen design. You run it on the DBLP records of the test split, which no step has used, against all of ACM.

In [ ]:
def link_records(A, B, S_tfidf, S_embed, model, counts, lower=LOWER, upper=UPPER, k=K):
    """The whole design on two prepared sources: block, compare, score, one-to-one, three bands."""
    pairs = pd.concat([top_k(S_tfidf, A, B, k), top_k(S_embed, A, B, k)]).drop_duplicates().reset_index(drop=True)
    v = compare(pairs, A, B, S_tfidf, S_embed)
    v["title_shared"] = title_shared(pairs, A, B, counts)
    score = one_to_one(pairs, model.predict_proba(v[FIELDS2])[:, 1])
    out = pairs.assign(score=score.round(3), decision=np.select([score >= upper, score >= lower], ["accept", "review"], "no match"))
    return out[out.decision != "no match"].sort_values("score", ascending=False).reset_index(drop=True)

In [ ]:
# New input: the test records of DBLP, which no step has used, linked against all of ACM.
new_dblp = dblp[dblp.id.map(lambda x: split["a" + x] == "test")]
A_new = prepare(new_dblp)
rows_new = A.id.isin(new_dblp.id).to_numpy()
t0 = time.perf_counter()
run = link_records(A_new, B, S_tfidf[rows_new], S_embed[rows_new], lgbm2, title_count)
run_seconds = time.perf_counter() - t0
run = with_labels(run, matches, split)
print(f"{len(new_dblp)} new records, {run_seconds:.1f} s:", run.decision.value_counts().to_dict())
print(run.groupby("decision").same.agg(["count", "sum"]).rename(columns={"sum": "true"}).to_string())

The pairs that go to a person:

In [ ]:
show = run.assign(DBLP=run.id_a.map(names_a.title), ACM=run.id_b.map(names_b.title))
print(show[show.decision == "review"][["score", "same", "DBLP", "ACM"]].to_string(index=False))

> **Check.** Almost the whole review queue is one title: "Book Review Column". Every pair has the same score, because the records are the same in every field. A person needs the issue or the pages to decide, from the library's web page.

## 12. Recap

- On clean records, normalising and one rule (title, year, venue) already reach F1 0.93.
- Splink without labels works well here, because the fields are clean and separate.
- LightGBM is better still, but its false links come from titles that many papers share.
- Error analysis found the cause; one new field (how common the title is) removed every false link in the test split.
- The decision: accept above the upper threshold, review the band, and know which records no field can decide.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Find duplicate people and companies](https://learning.nextia-ai.com/courses/record-matching/m09/find-duplicate-people-and-companies/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/record-matching/m09/link-bibliographic-records/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The records are real bibliographic records from DBLP and the ACM Digital Library, published by the University of Leipzig under CC BY 4.0.